In [23]:
import requests
from bs4 import BeautifulSoup
import json
import sys
import os
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from elasticsearch import Elasticsearch, helpers
import math
from typing import List

In [24]:
# fetch_to_json
def main():
    # 명령줄 인자 있으면 그걸 사용, 없으면 data/url.txt 읽기
    if len(sys.argv) > 1:
        urls = sys.argv[1:]
        print(f"[명령줄 인자] {len(urls)}개 url 입력받음")
        url_tuples = [(u, None) for u in urls]
    else:
        data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
        url_txt = os.path.join(data_dir, 'url.txt')
        if not os.path.exists(url_txt):
            print("사용법: python fetch_to_json.py url1 url2 ...  또는  data/url.txt 파일에 url 한 줄씩 입력")
            sys.exit(1)
        url_tuples = []
        with open(url_txt, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                if '#' in line:
                    url_part, title_part = line.split('#', 1)
                    url = url_part.strip()
                    title = title_part.strip()
                    if url:
                        url_tuples.append((url, title))
                else:
                    url_tuples.append((line, None))
        print(f"[data/url.txt] {len(url_tuples)}개 url 읽음")
    results = []
    for url_tuple in url_tuples:
        print(f"수집 중: {url_tuple[0]}")
        doc = fetch_content(url_tuple)
        if doc:
            results.append(doc)
    data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
    os.makedirs(data_dir, exist_ok=True)
    result_json_path = os.path.join(data_dir, 'result.json')
    with open(result_json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"총 {len(results)}개 문서 저장 완료: {result_json_path}")

In [25]:
# index_to_elasticsearch
# Elasticsearch 연결 (보안을 껐으므로 주소만 입력)
es = Elasticsearch("http://localhost:9200")

# ../data/result.json에서 데이터 읽기
data_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'result.json'))
print(data_path)
with open(data_path, 'r', encoding='utf-8') as f:
    docs = json.load(f)

index_name = "korean_news"

# bulk 색인을 위한 데이터 형식으로 변환
# generator를 사용하여 메모리를 효율적으로 사용
def generate_actions(documents):
    for i, doc in enumerate(documents):
        yield {
            "_index": index_name,
            "_id": i + 1,
            "_source": doc
        }

try:
    success, failed = helpers.bulk(es, generate_actions(docs), stats_only=True)
    print(f"성공적으로 색인된 문서 수: {success}")
    if failed:
        print(f"색인 실패한 문서 수: {failed}")
except Exception as e:
    print(f"Bulk 색인 중 오류 발생: {e}")

print("모든 데이터 색인을 완료했습니다!")

/Users/hyeonseong/workspace/BigDataInformationSearch/data/result.json
성공적으로 색인된 문서 수: 109
모든 데이터 색인을 완료했습니다!


In [26]:
# 여러 검색어에 대해 BM25 검색 결과 출력

es = Elasticsearch("http://localhost:9200")
index_name = "korean_news"

search_query_list = ["AI", "검색 엔진", "정치", "자동화", "인재"]  # 여러 쿼리 리스트

for search_query in search_query_list:
    res = es.search(
        index=index_name,
        query={
            "match": {
                "content": search_query
            }
        },
        # query={
        #     "match_all": {
        #     }
        # },
        # size=109  # 상위 5개 결과만 가져오기
    )

    print(f"'{search_query}' 검색 결과 (상위 5개):")
    print("-" * 50)
    for hit in res['hits']['hits'][:5]:
        doc_id = hit['_id']
        score = hit['_score']
        title = hit['_source'].get('title', '')
        print(f"_id: {doc_id} | 점수: {score:.2f}  |  제목: {title}")
    print("-" * 50)

'AI' 검색 결과 (상위 5개):
--------------------------------------------------
_id: 16 | 점수: 3.10  |  제목: SKT, AI 사업 총괄할 CIC 출범…2030년 연매출 5조 달성 목표
_id: 100 | 점수: 3.05  |  제목: 오픈AI, '데브데이 2025'서 비공개 프로젝트 공개한다
_id: 105 | 점수: 3.05  |  제목: [AI 한쿡] 조 단위 몸값 증명한 AI반도체... 영상이해도 MCP 시대
_id: 14 | 점수: 3.03  |  제목: “AI는 디지털 동료” MS, 국내 기업 AI에이전트 도입 사례 공개
_id: 12 | 점수: 3.03  |  제목: 과기정통부, 연구개발특구 내 'AI 글로벌 빅테크' 키운다
--------------------------------------------------
'검색 엔진' 검색 결과 (상위 5개):
--------------------------------------------------
_id: 14 | 점수: 4.07  |  제목: “AI는 디지털 동료” MS, 국내 기업 AI에이전트 도입 사례 공개
_id: 27 | 점수: 3.82  |  제목: 자격증도 없는데 뭘… 자격체계 없는 신직업에 쏟아진 '무관심'
_id: 11 | 점수: 3.57  |  제목: Chrome의 새로운 AI 기능들
_id: 34 | 점수: 3.26  |  제목: "우리가 찰리 커크다"... 그게 가능한가? [신필규의 아직도 적응 중]
_id: 4 | 점수: 3.20  |  제목: AI가 주니어를 빛나게 할 것이라 했지만, 왜 시니어만 더 강해졌을까?
--------------------------------------------------
'정치' 검색 결과 (상위 5개):
--------------------------------------------------
_id: 50 | 점수: 4.58  |  제목: 민주당 "권성동, 반성과 사죄가 먼저‥정

In [27]:
def precision_at_k(relevant: dict, retrieved: list, k: int) -> float:
    """
    Precision@K: 상위 K개 검색 결과 중 relevance_level이 1 이상인 문서의 비율
    relevant: {문서id: relevance_level, ...}
    """
    if k == 0:
        return 0.0
    hit_count = sum([1 for doc_id in retrieved[:k] if relevant.get(doc_id, 0) > 0])
    return hit_count / k
def average_precision(relevant: dict, retrieved: list, k: int) -> float:
    """
    Average Precision: 검색 결과에서 relevance_level이 1 이상인 문서가 등장할 때마다의 Precision의 평균
    """
    score = 0.0
    hit_count = 0
    for i, doc_id in enumerate(retrieved[:k]):
        if relevant.get(doc_id, 0) > 0:
            hit_count += 1
            score += hit_count / (i + 1)
    num_relevant = sum([1 for v in relevant.values() if v > 0])
    return score / min(num_relevant, k) if num_relevant else 0.0
def mean_average_precision(relevant_lists: list, retrieved_lists: list, k: int) -> float:
    """
    MAP: 여러 쿼리의 Average Precision의 평균
    """
    scores = [average_precision(rel, ret, k) for rel, ret in zip(relevant_lists, retrieved_lists)]
    return sum(scores) / len(scores) if scores else 0.0
def mean_reciprocal_rank(relevant_lists: list, retrieved_lists: list) -> float:
    """
    MRR: 첫 번째로 relevance_level이 1 이상인 정답의 순위의 역수 평균
    """
    rr_scores = []
    for relevant, retrieved in zip(relevant_lists, retrieved_lists):
        rank = 0
        for i, doc_id in enumerate(retrieved):
            if relevant.get(doc_id, 0) > 0:
                rank = i + 1
                break
        rr_scores.append(1 / rank if rank > 0 else 0.0)
    return sum(rr_scores) / len(rr_scores) if rr_scores else 0.0
def ndcg_at_k(relevant: dict, retrieved: list, k: int) -> float:
    """
    nDCG@K: 순위별로 정답의 중요도를 반영한 점수 (graded relevance)
    relevant: {문서id: relevance_level, ...}
    """
    import math
    def dcg(rels):
        return sum([rel / math.log2(i + 2) for i, rel in enumerate(rels)])
    rels = [relevant.get(doc_id, 0) for doc_id in retrieved[:k]]
    ideal_rels = sorted(relevant.values(), reverse=True)[:k]
    ideal_rels += [0] * (k - len(ideal_rels))
    dcg_score = dcg(rels)
    idcg_score = dcg(ideal_rels)
    return dcg_score / idcg_score if idcg_score > 0 else 0.0

In [28]:
# 검색 성능 평가 (Precision@K, MAP, MRR, nDCG)

es = Elasticsearch("http://localhost:9200")
index_name = "korean_news"
# 평가용 쿼리와 정답 로드
answer_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'ground_truth.json'))
with open(answer_path, 'r', encoding='utf-8') as f:
    answer_data = json.load(f)
# answer_data는 [{"query": ..., "answers": [...]}, ...] 형태라고 가정
queries = [item['query'] for item in answer_data]
relevant_lists = [item['answers'] for item in answer_data]
retrieved_lists = []
K = 5 # 평가할 K값
for query in queries:
    res = es.search(
        index=index_name,
        query={"match": {"content": query}},
        size=K
    )
    retrieved = [hit['_id'] for hit in res['hits']['hits']]
    retrieved_lists.append(retrieved)
# 평가 지표 계산
p_at_k = [precision_at_k(rel, ret, K) for rel, ret in zip(relevant_lists, retrieved_lists)]
map_score = mean_average_precision(relevant_lists, retrieved_lists, K)
mrr_score = mean_reciprocal_rank(relevant_lists, retrieved_lists)
ndcg_scores = [ndcg_at_k(rel, ret, K) for rel, ret in zip(relevant_lists, retrieved_lists)]
print(f"검색 성능 평가 결과!")
for i, query in enumerate(queries):
    print(f"\n[Query] {query}")
    print(f"Precision@{K}: {p_at_k[i]:.4f}")
    print(f"nDCG@{K}: {ndcg_scores[i]:.4f}")
print("\nMAP:", f"{map_score:.4f}")
print("MRR:", f"{mrr_score:.4f}")

검색 성능 평가 결과!

[Query] AI
Precision@5: 0.8000
nDCG@5: 0.4607

[Query] 검색 엔진
Precision@5: 0.8000
nDCG@5: 0.9020

[Query] 정치
Precision@5: 1.0000
nDCG@5: 1.0000

[Query] 투자
Precision@5: 0.6000
nDCG@5: 0.5915

[Query] 인재
Precision@5: 0.6000
nDCG@5: 0.6239

[Query] 추석
Precision@5: 0.2000
nDCG@5: 0.1536

MAP: 0.5856
MRR: 0.9167
